这份实验报告和代码的深度非常令人佩服。你通过严谨的 **TS (纯时序)** vs **CS (截面)** 对比，得出了一个在量化界极其经典且重要的结论：**纯绝对收益（Absolute Return）作为标签的信噪比（SNR）极低，模型很难从中提取有效的个股排序规律。**

### 1. 代码审查：技术逻辑无误
*   **实现严谨**：你正确地处理了跨日滚动均值（`self_mean_ret_20d`）、波动率标准化（`TS-B`）以及不同标签的 Rank IC 计算。
*   **回测对齐**：所有标签都在同一个 `LGB_PARAMS` 下训练，且统一使用了 2x 缓冲的换手控制，这使得对比结果具有极高的可信度。

---

### 2. 结果解读：为什么时序模型（TS-A/B/C）“全军覆没”？

你的分析非常准确：**“信噪比太低”**。我们可以从三个维度进一步深挖：

1.  **Beta 淹没了 Alpha**：
    在 120 分钟的维度，个股的 `raw_ret` 中，大盘（Beta）的波动贡献了约 80-90% 的能量。模型在训练时，90% 的精力都在试图预测下两小时大盘是涨是跌。然而，量价特征（X1/X2）对大盘择时的能力很弱，导致模型学不到东西（早停在 1-6 轮）。
2.  **“最差 5 天”的防御幻觉**：
    你在表 5 发现 TS-A/B 的预测值在最差 5 天也是正的。这说明模型学会了“在这个池子里，长期来看大家都是涨的”，它变成了一个**盲目的乐观主义者**，完全失去了区分个股好坏的能力。
3.  **均值回归 vs 截面排序**：
    老板要求的“和自己比”，本质上是在做**择时（Timing）**。而我们的策略是 Top-N 选股，本质上是在做**排序（Ranking）**。用择时模型去做排序，相当于让一个只会判断“天气冷不冷”的人去给“谁跑得快”排名，逻辑上存在错配。

---

### 3. 下一步思考：如何挽救老板的“时序思路”？

老板的思路在理论上是更高阶的：**“只有当一个个股表现出超越其自身常态的爆发力时，才买入。”**

要让这个思路 work，我们不能简单地预测绝对收益，而需要引入 **“二阶目标预测”**。

#### **核心建议：从“绝对回归”转向“极端事件分类（Breakout Classification）”**

时序逻辑在处理“连续值”时非常平庸，但在处理**“异动/突变”**时非常强大。

**新方案 V3：异动标签（Self-Z Labeling）**
*   **标签定义**：如果个股未来 2 小时的收益，超过了它自身过去 20 天平均波动的 1.5 倍标准差，标记为 1，否则为 0。
*   **意义**：不再让模型学习 5bps 和 10bps 的区别，而是让模型学习**“这只票是不是要爆发了”**。

---

### 4. 建议执行代码：异动概率模型（The TS-Breakout Model）

请尝试构建以下代码。它遵循老板的“和自己比”的思路，但采用了“分类器”来解决信噪比问题。

```python
"""
V3: 异动概率模型 (TS-Breakout Model)
逻辑：遵循老板"和自己比"的思路，预测个股未来2小时产生"超常波动"的概率
"""

def build_breakout_labels(df, sigma_threshold=1.0):
    logger.info(f"构建异动分类标签 (Threshold={sigma_threshold} sigma)...")
    
    # 1. 纯时序标准化：收益 / 历史波动率
    # 这一步完全是"和自己比"，不参考大盘
    df['std_ret'] = df['raw_ret'] / df['hist_vol_20d'].clip(lower=0.005)
    
    # 2. 定义二分类标签：是否发生了向上的超常波动
    # 相比回归，分类能有效过滤掉那些微小的、不可测的随机噪音
    df['label'] = (df['std_ret'] > sigma_threshold).astype(int)
    
    pos_ratio = df['label'].mean()
    logger.info(f"正样本占比: {pos_ratio:.2%}")
    return df

# 训练建议：
# 使用 Binary Objective
LGB_PARAMS_V3 = {
    "objective": "binary",
    "metric": "auc",
    "is_unbalance": True, # 因为异动是少数，必须处理样本不平衡
    "num_leaves": 15,     # 保持简单，防止过拟合
    "learning_rate": 0.03,
}
```

### 5. 关于“卖出逻辑”的降维打击

既然时序模型在“买入”上很难跑赢截面模型（因为截面天生就是为选股设计的），那么时序模型最好的舞台其实是**“动态止盈止损”**。

**一个非常有价值的下一步：**
*   **CS模型选股（CS-Best）**：负责 10:30 选出 Top-50。
*   **TS模型守仓（TS-Exit）**：针对这 50 只票，每一分钟跑一次 TS 模型。如果 TS 模型预测该个股的“爆发动能”已衰竭（例如 `raw_ret / vol` 开始掉头），则提前离场。

---

### 总结与分享

1.  **坚守 CS-Best**：目前你的 `0.7V+0.3C` 已经是极其优秀的基准（ExcBps +22, Sharpe 3.93），这在生产环境已经是可以实盘的水平。
2.  **回应老板**：不要直接否定时序，而是告诉他：“纯时序预测容易被大盘 Beta 误导，我们正在通过**‘异动概率识别’（Anomaly Detection）**来提取纯净的个体信号，并将其作为截面模型的**二阶过滤（Meta-Labeling）**。”
3.  **数据质量**：建议你在特征中加入 **`X1_zscore_self`**（即 X1 除以 X1 过去 20 天的标准差）。这能让模型更好地理解“目前的偏离对于这只股票来说算不算大”。

**你需要我帮你写一段将 CS 选股结果和 TS 异动过滤结合起来的“双模回测”脚本吗？这通常是量化团队最后的杀手锏。**